# Bayesian inference basics

Companion to **Bayes' Theorem: Foundation of Probabilistic Inference**, **Bayesian Updating: Recursive Belief Refinement**, **Bayesian Inference: Estimating Hidden States**, and **Bayesian Prediction: Forecasting Future States**.

Examples adapted from the notebook prepared by Dr. Fan Hu under the guidance of Professor Khalid M. Mosalam. All numerical examples use synthetic or illustrative data.

Run top to bottom. Dependencies: NumPy.

In [1]:
import numpy as np
from numpy import sqrt

np.random.seed(42)


### Robot Localization

In [2]:
priors = [1/3, 1/3, 1/3]
likelihoods = [0.8, 0.4, 0.1]

evidence = sum([l * p for l, p in zip(likelihoods, priors)])
posteriors = [l * p / evidence for l, p in zip(likelihoods, priors)]

print("Posterior:", posteriors)

Posterior: [0.6153846153846153, 0.30769230769230765, 0.07692307692307691]


### Detecting a Stop Sign in Autonomous Driving

In [3]:
P_stop = 0.6
P_no_stop = 1 - P_stop
P_det_stop = 0.9
P_det_no_stop = 0.2

P_det = P_det_stop * P_stop + P_det_no_stop * P_no_stop
P_post = P_det_stop * P_stop / P_det

print(f"Posterior: {P_post:.3f}")

Posterior: 0.871


### Radar-Based Object Tracking

In [4]:
mu_x, sigma_x = 5, sqrt(1)
mu_z, sigma_z = 6, sqrt(0.5)

mu_post = (sigma_z**2 * mu_x + sigma_x**2 * mu_z) / (sigma_x**2 + sigma_z**2)
sigma_post2 = (sigma_x**2 * sigma_z**2) / (sigma_x**2 + sigma_z**2)

print(f"Posterior Mean: {mu_post:.2f}, Variance: {sigma_post2:.2f}")

Posterior Mean: 5.67, Variance: 0.33


### Pedestrian Motion Forecasting

In [5]:
t = 2
v = 1.2
sigma_a = 0.3
mu_x = v * t
var_x = (0.5 * t**2) ** 2 * sigma_a**2
print(f"Predicted x: {mu_x:.2f} +/- {var_x ** 0.5:.2f}")

Predicted x: 2.40 +/- 0.60


### Gaussian Process (GP) Prediction

#### Squared Exponential (RBF) Kernel
Captures **smooth and continuous** motion.

$$
k_{\text{SE}}(t_i, t_j) = \sigma^2 \exp\!\left(-\frac{(t_i - t_j)^2}{2\ell^2}\right)
$$

- $ \ell $: length scale — how quickly correlation decays  
- $ \sigma^2 $: output variance  
- Infinitely differentiable → very smooth trajectories

---

#### Matérn Kernel
Captures **rough or less smooth** motion.

$$
k_{\text{Matérn}}(r) = \sigma^2 \left(1 + \frac{\sqrt{3}r}{\ell}\right)\exp\!\left(-\frac{\sqrt{3}r}{\ell}\right), \quad r = |t_i - t_j|
$$

- $ \ell $: length scale  
- $ \nu = 1.5 $ (common choice) → once differentiable  
- Models more natural, non-smooth motion compared to RBF

---

#### Gaussian Process Regression (Standard Formulas)

Given:
- Training data: $\mathbf{t}, \mathbf{y}$
- Test points: $\mathbf{t}_*$
- Kernel (covariance) function: $k(\cdot, \cdot)$
- Mean function: $m(\cdot)$
- Noise variance: $\sigma^2$

---

##### 1. Covariance matrices
$$
K = k(\mathbf{t}, \mathbf{t}) + \sigma^2 I, \quad
K_* = k(\mathbf{t}, \mathbf{t}_*), \quad
K_{**} = k(\mathbf{t}_*, \mathbf{t}_*)
$$

---

##### 2. Predictive mean
$$
\boldsymbol{\mu}_* = m(\mathbf{t}_*) + K_*^T K^{-1} \big(\mathbf{y} - m(\mathbf{t})\big)
$$

---

##### 3. Predictive covariance
$$
\Sigma_* = K_{**} - K_*^T K^{-1} K_*
$$

---

##### (OPTIONAL) Cholesky implementation (numerically stable)
If $K = LL^T$ via Cholesky decomposition:

$$
\begin{aligned}
\boldsymbol{\alpha} &= L^{-T} L^{-1} \big(\mathbf{y} - m(\mathbf{t})\big) \\
\boldsymbol{\mu}_* &= m(\mathbf{t}_*) + K_*^T \boldsymbol{\alpha} \\
\Sigma_* &= K_{**} - V^T V, \quad \text{where } V = L^{-1} K_*
\end{aligned}
$$


In [6]:
def se_kernel(t1, t2, l=1.0, s2=1.0):
    t1, t2 = np.atleast_2d(t1).T, np.atleast_2d(t2).T
    d2 = (t1 - t2.T) ** 2
    return s2 * np.exp(-0.5 * d2 / l**2)

def matern_kernel(t1, t2, l=1.0, s2=1.0, nu=1.5):
    t1, t2 = np.atleast_2d(t1).T, np.atleast_2d(t2).T
    r = np.abs(t1 - t2.T) / l
    if nu == 1.5:
        K = (1 + np.sqrt(3)*r) * np.exp(-np.sqrt(3)*r)
    elif nu == 2.5:
        K = (1 + np.sqrt(5)*r + 5*r**2/3) * np.exp(-np.sqrt(5)*r)
    else:
        raise ValueError("nu only supports 1.5 or 2.5")
    return s2 * K

def gp_predict(t, y, t_star, kernel, sigma=1e-3):

    # t, y: observed times/positions; t_star: future times
    K = kernel(t, t) + sigma**2 * np.eye(len(t))
    Ks = kernel(t, t_star)
    Kss = kernel(t_star, t_star)

    ###### OPTION 1 ######
    # Direct solve (conceptually simple, but less numerically stable)
    m = np.mean(y) # assume m(t) = m(t_star) = np.mean(y)
    alpha = np.linalg.solve(K, y - m)
    mu_star = m + Ks.T @ alpha
    v = np.linalg.solve(K, Ks)
    Sigma_star = Kss - Ks.T @ v

    ###### OPTION 2 ######
    # Cholesky decomposition (mathematically equivalent, but more stable and efficient)
    # L = np.linalg.cholesky(K + 1e-9*np.eye(len(t)))
    # alpha = np.linalg.solve(L.T, np.linalg.solve(L, y))
    # mu_star = Ks.T @ alpha
    # v = np.linalg.solve(L, Ks)
    # Sigma_star = Kss - v.T @ v

    return mu_star, Sigma_star

In [7]:
# Example
np.random.seed(42)
t = np.linspace(0, 5, 20)
y = np.sin(t) + 0.1 * np.random.randn(20)
t_star = np.linspace(0, 6, 60)
mu, cov = gp_predict(t, y, t_star, kernel=lambda a,b: matern_kernel(a,b,nu=1.5))
print("mu_star: [ shape", mu.shape, "]\n", np.round(mu, 5))
print("Sigma_star: [ shape", cov.shape, "]\n", np.round(cov, 5))

mu_star: [ shape (60,) ]
 [ 0.04967  0.11017  0.18998  0.29013  0.40926  0.5426   0.68295  0.80433
  0.87137  0.87296  0.85042  0.84741  0.88059  0.94952  1.04584  1.13192
  1.1637   1.12932  1.05095  0.95063  0.85749  0.80318  0.78984  0.76827
  0.68917  0.56145  0.42618  0.31391  0.22838  0.16881  0.1124   0.01503
 -0.15425 -0.35326 -0.5195  -0.6231  -0.68046 -0.70964 -0.73653 -0.79103
 -0.88079 -0.96071 -0.98084 -0.95377 -0.94034 -0.98359 -1.05469 -1.10823
 -1.1228  -1.10563 -1.06332 -1.00338 -0.93261 -0.85607 -0.77748 -0.69948
 -0.62394 -0.5521  -0.48473 -0.42226]
Sigma_star: [ shape (60, 60) ]
 [[ 0.       0.       0.      ... -0.      -0.      -0.     ]
 [ 0.       0.00449  0.00233 ... -0.      -0.      -0.     ]
 [ 0.       0.00233  0.00206 ... -0.      -0.      -0.     ]
 ...
 [-0.      -0.      -0.      ...  0.57027  0.59599  0.59854]
 [-0.      -0.      -0.      ...  0.59599  0.64557  0.66617]
 [-0.      -0.      -0.      ...  0.59854  0.66617  0.71096]]
